# Ciena MCP paso a paso: NSI, inventario y métricas PM

Este notebook explica, con ejemplos ejecutables, los conceptos clave de la API
del Ciena MCP (Manage, Control and Plan) que usamos en `mcp_client/`.

Vamos a ver, en orden:

1. Qué es el MCP y sus servicios (`tron`, `nsi`, `pm`)
2. **Network Construct**: el objeto raíz de inventario
3. **Equipment**: las tarjetas/módulos físicos dentro de un NE
4. **PM Metrics**: cómo se consultan métricas de desempeño (facility, parameter, granularidad)

Cada sección combina una explicación corta con una llamada real a la API para
que veas la forma exacta de los datos.


## 1. ¿Qué es el MCP y por qué hay varios servicios?

El **Ciena MCP** (Manage, Control and Plan) es la plataforma de orquestación
de Ciena: un solo "gateway" HTTPS que internamente reparte las peticiones
entre varios microservicios según el prefijo de la URL. Los que usa esta
aplicación:

- **`tron`** (`/tron/api/v1/...`): servicio de **seguridad/autenticación**.
  Aquí se hace login (`POST /tron/api/v1/tokens`) y se obtiene el *bearer
  token* que se manda en el header `Authorization` de todas las demás
  llamadas. También administra usuarios, roles y sesiones activas.
- **`nsi`** (`/nsi/api/v7/...`): **Network Service Inventory** — el
  inventario de la red: qué elementos de red (NEs) existen, qué equipo físico
  tienen dentro, cómo están conectados.
- **`pm`** (`/pm/api/v3/...`): **Performance Monitoring** — series de tiempo
  con métricas de desempeño (potencia óptica, errores, etc.) recolectadas de
  cada NE.

Nuestro `MCPClient` (en `mcp_client/client.py`) se loguea una sola vez contra
`tron`, cachea el token, y lo reutiliza para llamar a `nsi` y `pm`.


In [ ]:
from mcp_client import MCPClient, NSIService, PMService

client = MCPClient()
nsi = NSIService(client)
pm = PMService(client)

print("Cliente listo. Token activo:", bool(client.auth.get_token()))


## 2. Network Construct: el objeto raíz del inventario

Un **network construct** es la representación, dentro del MCP, de un
**elemento de red (Network Element / NE)**: un equipo físico real
(un ROADM, un transponder, un switch óptico, etc.) que el MCP está
gestionando. Es el nodo topológico de más alto nivel — todo lo demás
(equipment, puertos, facilities) cuelga de un network construct.

Cada network construct trae, entre otros campos:

- `id`: identificador interno usado para pedir sub-recursos (equipment, etc.)
- `attributes.name`: nombre corto del NE (el que se usa como
  `networkElementName` al consultar métricas PM)
- `attributes.longName`: nombre largo/técnico
- `attributes.displayData.displayName`: nombre "amigable" que se muestra en
  la UI del MCP

Vamos a listar los network constructs disponibles:


In [ ]:
constructs = nsi.network_constructs()
print(f"{len(constructs)} network constructs encontrados\n")

for i, c in enumerate(constructs):
    # if i == 0: print(c)
    if "10.20." in c["attributes"].get("ipAddress", ""):
        a = c["attributes"]
        print(f"- name={a['name']!r}  displayName={a['displayData']['displayName']!r}  IpAddress={a['ipAddress']!r} id={c['id']}  ")

Tomemos el primero para explorarlo a fondo. Nota cómo la respuesta sigue el
formato **JSON:API** (`{"data": [...]}`, cada item con `id`, `type` y
`attributes`) — es el formato que usa toda la familia de APIs del MCP.


In [ ]:
import json

ne0 = constructs[0]
print(json.dumps(ne0, indent=2, ensure_ascii=False)[:1500])


## 3. Equipment: lo que hay físicamente dentro de un NE

Un network construct (el NE) es, a nivel físico, un chasis con **shelves**
(bandejas), **slots** (ranuras) y **tarjetas/módulos** insertados en esas
ranuras. El recurso `equipment` de `nsi` lista justamente esas piezas físicas
que existen *dentro* de un NE específico.

Ejemplos típicos de `displayName` que vas a encontrar:
- **TRM**: *Transponder/Transceiver Module* — la tarjeta que convierte señal
  óptica ⇄ eléctrica en un puerto de línea.
- Tarjetas de switching, controladoras, fuentes de poder, etc.

Pedimos el equipment del NE que elegimos arriba:


In [ ]:
equipment = nsi.equipment(ne0["id"])
print(f"{len(equipment)} piezas de equipment en {ne0['attributes']['name']}\n")

for i,e in enumerate(equipment):
    if i == 0: print(e['attributes'])
    print(f"- displayName={e['attributes']['displayData']['displayName']!r} serialNumber={e['attributes'].get('installedSpec', {}).get('serialNumber', '')} partNumber={e['attributes'].get('installedSpec', {}).get('partNumber', '')}  id={e['id']}  ")


In [ ]:
trm_equipment = [e for e in equipment if "TRM" in e["attributes"]["displayData"]["displayName"]]
print(f"{len(trm_equipment)} tarjetas TRM (transponder) encontradas")
for i, e in enumerate(trm_equipment):
    if i == 0: print(e['attributes'])
    print(" -", e["attributes"]["displayData"]["displayName"], "| locations:", e["attributes"].get("locations"), "| State:", e["attributes"].get("state"), "| serialNumber:", e["attributes"].get("installedSpec", {}).get("serialNumber"), "| partNumber:", e["attributes"].get("installedSpec", {}).get("partNumber"), "| type:", e["attributes"].get("installedSpec", {}).get("type"))


## 4. Facility Resources (`fres`): enlaces y etiquetas puestas por humanos

Hasta ahora vimos objetos que describen **qué hay** dentro de un NE (network
construct, equipment). Pero muchas veces lo que quieres saber es **con qué
está conectado** ese NE — un enlace de fibra hacia otro sitio, un patch cord
hacia un cliente, etc. Y ahí suele haber una etiqueta que puso una persona
para documentar el otro extremo (ej. "hacia el sitio X, puerto Y").

Ese objeto es **`fres`** (*Facility Resource Endpoints*), en
`/nsi/api/v7/fres`. A diferencia de `equipment` (una tarjeta física) o de un
`facilityNameNative`/PTP de PM (un puerto puntual), un `fre` representa un
**enlace entre dos puntos** — pueden estar en el mismo NE (ej. la fibra
interna entre un mux/demux y un amplificador) o en **dos NEs distintos**
(un enlace de línea hacia otro sitio).

Campos importantes:

- **`userLabel`**: la etiqueta de texto libre que alguien escribió a mano
  (ej. `"TO SITE-B PATCH-042"`, `"LINK CARRIER-X CIRCUIT 12345"`).
- **`note.noteMsg`**: una anotación adicional, con `lastUpdatedBy` y
  `lastUpdatedTime` — quién la escribió y cuándo.
- **`linkLabel`** / **`serviceLabel`**: identifica **ambos extremos** del
  enlace, con el formato `"NE1:shelf-slot-port,NE2:shelf-slot-port"`. Aquí es
  donde ves el equipo del otro extremo — que puede ser un NE completamente
  distinto.
- **`layerRate`**: la capa del enlace (`PHY` = fibra física, `OTS` = óptico
  de línea, `DSR_ETHERNET`/`RS` = enlace de cliente, etc.), útil para saber
  si es un enlace de amplificador/línea o de transponder/cliente.

Nota de la llamada a la API: igual que `equipment`, requiere el filtro
`networkConstruct.id` — pero a diferencia de otros recursos, **no acepta**
`pageSize` como query param (devuelve `400 INV-201` si lo mandas).


In [ ]:
fres = nsi.facility_resources(ne0["id"])
print(f"{len(fres)} facility resources (enlaces) tocan a {ne0['attributes']['name']}\n")

etiquetados = [f for f in fres if f["attributes"].get("userLabel")]
print(f"{len(etiquetados)} tienen userLabel puesto por un humano:\n")

print(fres[0])

for f in etiquetados:
    a = f["attributes"]
    otro_extremo = a.get("linkLabel") or a.get("serviceLabel") or ""
    print(f"- userLabel={a['userLabel']!r}  layerRate={a.get('layerRate')}  extremos={otro_extremo!r}")


In [ ]:
import json

if etiquetados:
    print(json.dumps(etiquetados[0], indent=2, ensure_ascii=False))


## 5. PM Metrics: cómo se mide el desempeño

El servicio `pm` guarda **series de tiempo** de métricas recolectadas
periódicamente de cada NE. Para consultarlas necesitas identificar *qué*
punto exacto del equipo estás midiendo. Los conceptos clave:

- **`networkElementName`**: el `name` del network construct (NE) — el mismo
  que vimos en la sección 2.
- **`facilityNameNative`**: identifica el punto físico/lógico dentro del NE
  donde se toma la medición. Sigue un patrón tipo `PTP-<shelf>-<slot>-<port>`.
  **PTP** = *Physical Termination Point* (el punto físico de terminación de
  una señal, ej. un puerto óptico concreto). Ejemplo: `PTP-1-3-1` = shelf 1,
  slot 3, puerto 1.
- **`parameter`**: qué magnitud se mide en ese punto, ej. `RX_OPTICAL_POWER`
  (potencia óptica recibida, en dBm).
- **`range`**: ventana de tiempo a consultar — puede ser relativa
  (`{"type": "relative", "unit": "HOURS", "value": 1}` = "la última hora") o
  absoluta (con `startTime`/`endTime`).
- **`granularity`**: cada cuánto viene un dato dentro de esa ventana (ej.
  `15_MINUTE` = un dato cada 15 minutos).

Consultemos la potencia óptica recibida del PTP-1-3-1 de nuestro NE en la
última hora:


In [ ]:
metrics = pm.query_metrics(
    network_element_name=ne0["attributes"]["name"],
    facility_name_native="PTP-1-3-1",
    parameter="RX_OPTICAL_POWER",
    range_unit="HOURS",
    range_value=1,
)

print(f"{len(metrics)} series de métricas devueltas\n")
print(json.dumps(metrics[0], indent=2, ensure_ascii=False))


Fíjate en la forma de cada elemento de `metrics`:

- `attributes.tags`: metadatos que identifican **qué** se midió — el mismo
  NE/facility/parameter que pediste, más contexto (`shelf`, `slot`, `port`,
  `unit`, `granularity`, `direction` -RECEIVE/TRANSMIT-, etc.)
- `attributes.values`: un diccionario donde **cada clave es un timestamp**
  (ISO 8601) y el valor es `{"value": <número>, "condition": "OK"/... }` — un
  punto de la serie de tiempo, uno por cada intervalo de `granularity`
  dentro del rango pedido.

Vamos a extraer solo los pares (timestamp, valor) para verlo más claro:


In [ ]:
for ts, v in metrics[0]["attributes"]["values"].items():
    print(f"{ts}  ->  {v['value']} {metrics[0]['attributes']['tags']['unit']}  ({v['condition']})")


## Resumen de conceptos

| Concepto | Qué es | Dónde lo ves |
|---|---|---|
| Network Construct | Un elemento de red (NE) completo, ej. un ROADM | `nsi.network_constructs()` |
| Equipment | Tarjeta/módulo físico dentro de un NE (ej. TRM) | `nsi.equipment(ne_id)` |
| Facility / PTP | Puerto o punto físico concreto dentro de un equipo | `facilityNameNative` en PM |
| Parameter | Magnitud medida en esa facility (ej. potencia óptica) | `parameter` en PM |
| Metric series | Serie de tiempo de un parameter en una facility | `pm.query_metrics(...)` |\n| Facility Resource (fre) | Enlace entre dos puntos (mismo NE u otro NE), con `userLabel` puesto a mano | `nsi.facility_resources(ne_id)` |

## Próximos pasos

- Explora otros `parameter` disponibles para el mismo facility (errores,
  temperatura, etc.) — puedes pedirle a `test1.py`/`main.py` que imprima
  parámetros distintos.
- Explora otros NEs (`constructs[1]`, `constructs[2]`, ...) y compara su
  equipment.
- Si quieres, podemos agregar al `mcp_client` un wrapper para el servicio
  `tron` (usuarios, roles, sesiones) o para *fault management*, y hacer un
  notebook similar para esos conceptos.
